In [1]:
library(affy)
library(GEOquery)
library(tidyverse)

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: 'generics'


The following objects are masked from 'package:base':

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: 'BiocGenerics'


The following objects are masked from 'package:stats':

    IQR, mad, sd, var, xtabs


The following objects are masked from 'package:base':

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Bioba

In [2]:
# defining datasets
training_set   <- "GSE42568"
validation_set <- c("GSE21653", "GSE20711", "GSE88770")

curr <- validation_set[3]

In [3]:
# reading in .cel files
raw_data <- ReadAffy(celfile.path = paste0("../data/cel_files/", curr))

# RMA normalize data
normalized_data <- rma(raw_data)

# get expression data
normalized_expr <- as.data.frame(exprs(normalized_data))
normalized_expr <- tibble::rownames_to_column(normalized_expr, var = "ID")

# Clean column names - extract just GSM IDs
colnames(normalized_expr) <- gsub("_.*", "", colnames(normalized_expr))

head(normalized_expr)


Warning message:
"replacing previous import 'AnnotationDbi::head' by 'utils::head' when loading 'hgu133plus2cdf'"
Warning message:
"replacing previous import 'AnnotationDbi::tail' by 'utils::tail' when loading 'hgu133plus2cdf'"




Background correcting
Normalizing
Calculating Expression


,ID,GSM2345257,GSM2345258,GSM2345259,GSM2345260,GSM2345261,GSM2345262,GSM2345263,GSM2345264,GSM2345265,⋯,GSM2345364,GSM2345365,GSM2345366,GSM2345367,GSM2345368,GSM2345369,GSM2345370,GSM2345371,GSM2345372,GSM2345373
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1007_s_at,10.681277,10.802657,11.001365,10.040305,10.248975,9.956543,10.785764,10.681655,10.821716,⋯,10.449581,10.454548,10.466526,10.538790,10.488697,10.464901,9.954742,11.106150,11.135586,11.383322
2,1053_at,8.159738,7.527161,8.331114,7.640328,7.372783,7.530817,8.031239,7.956069,7.410529,⋯,7.570383,8.143535,7.931239,7.532532,7.660106,7.914382,7.588485,7.839713,8.298097,7.229432
3,117_at,6.093081,6.218630,6.123202,7.689468,6.004692,6.651521,6.365263,5.896828,5.974322,⋯,5.675094,6.241452,9.372554,6.038132,6.186391,5.329259,5.537211,5.699836,6.461560,6.571032
4,121_at,7.778084,8.301535,7.564411,7.632244,7.835956,7.590282,7.275567,7.732639,7.913801,⋯,7.157018,7.600315,7.445740,7.648995,7.126868,7.237025,7.460775,7.868601,7.031209,7.894587
5,1255_g_at,3.150079,3.190410,3.020225,2.857195,2.875915,3.062885,2.885085,2.770437,2.817571,⋯,2.933883,2.841175,2.814711,2.756995,2.879981,2.813866,2.776217,2.878176,2.996537,2.996122
6,1294_at,6.712948,6.920108,7.766302,8.029146,7.707253,7.997286,7.763012,8.049655,7.774498,⋯,8.147020,7.771451,7.708887,8.232936,7.893129,7.177375,7.536349,7.839510,7.670078,8.223649


In [4]:
# map probe IDs to gene symbols
gse <- getGEO(curr, GSEMatrix = TRUE)

Found 1 file(s)

GSE88770_series_matrix.txt.gz



In [5]:
# fetch feature data to get ID - gene symbol mapping
feature_data <- gse[[paste0(curr, "_series_matrix.txt.gz")]]@featureData@data

# subset
feature_data <- feature_data[c(1, 11)]

head(feature_data)

,ID,Gene Symbol
,<chr>,<chr>
1007_s_at,1007_s_at,DDR1 /// MIR4640
1053_at,1053_at,RFC2
117_at,117_at,HSPA6
121_at,121_at,PAX8
1255_g_at,1255_g_at,GUCA1A
1294_at,1294_at,MIR5193 /// UBA7


In [6]:
normalized_expr <- normalized_expr |>
  inner_join(feature_data, by = "ID")

head(normalized_expr)
dim(normalized_expr)

,ID,GSM2345257,GSM2345258,GSM2345259,GSM2345260,GSM2345261,GSM2345262,GSM2345263,GSM2345264,GSM2345265,⋯,GSM2345365,GSM2345366,GSM2345367,GSM2345368,GSM2345369,GSM2345370,GSM2345371,GSM2345372,GSM2345373,Gene Symbol
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,1007_s_at,10.681277,10.802657,11.001365,10.040305,10.248975,9.956543,10.785764,10.681655,10.821716,⋯,10.454548,10.466526,10.538790,10.488697,10.464901,9.954742,11.106150,11.135586,11.383322,DDR1 /// MIR4640
2,1053_at,8.159738,7.527161,8.331114,7.640328,7.372783,7.530817,8.031239,7.956069,7.410529,⋯,8.143535,7.931239,7.532532,7.660106,7.914382,7.588485,7.839713,8.298097,7.229432,RFC2
3,117_at,6.093081,6.218630,6.123202,7.689468,6.004692,6.651521,6.365263,5.896828,5.974322,⋯,6.241452,9.372554,6.038132,6.186391,5.329259,5.537211,5.699836,6.461560,6.571032,HSPA6
4,121_at,7.778084,8.301535,7.564411,7.632244,7.835956,7.590282,7.275567,7.732639,7.913801,⋯,7.600315,7.445740,7.648995,7.126868,7.237025,7.460775,7.868601,7.031209,7.894587,PAX8
5,1255_g_at,3.150079,3.190410,3.020225,2.857195,2.875915,3.062885,2.885085,2.770437,2.817571,⋯,2.841175,2.814711,2.756995,2.879981,2.813866,2.776217,2.878176,2.996537,2.996122,GUCA1A
6,1294_at,6.712948,6.920108,7.766302,8.029146,7.707253,7.997286,7.763012,8.049655,7.774498,⋯,7.771451,7.708887,8.232936,7.893129,7.177375,7.536349,7.839510,7.670078,8.223649,MIR5193 /// UBA7


[1] 54675   119

In [7]:
# For multiple probes → one gene: take average
# For one probe → multiple genes: delete

normalized_expr <- normalized_expr |>
  # Remove rows where gene symbol is empty or maps to multiple genes
  filter(!grepl("///", `Gene Symbol`)) |>
  filter(`Gene Symbol` != "") |>

  # Group by gene symbol and average across probes
  group_by(`Gene Symbol`) |>
  summarise(across(where(is.numeric), mean)) |>
  ungroup() |>
  rename(gene_symbol = `Gene Symbol`)

dim(normalized_expr)
head(normalized_expr)

[1] 21655   118

gene_symbol,GSM2345257,GSM2345258,GSM2345259,GSM2345260,GSM2345261,GSM2345262,GSM2345263,GSM2345264,GSM2345265,⋯,GSM2345364,GSM2345365,GSM2345366,GSM2345367,GSM2345368,GSM2345369,GSM2345370,GSM2345371,GSM2345372,GSM2345373
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
A1BG,6.823343,6.841661,7.517482,7.066259,7.407212,7.356098,7.154821,7.096108,6.468802,⋯,6.976417,5.859655,7.837866,6.895554,7.250273,6.311108,6.544339,6.910726,6.550731,5.887219
A1BG-AS1,5.380914,4.960461,5.632821,5.161617,5.438206,5.880592,5.756559,5.504966,5.310122,⋯,5.234477,4.738434,5.330613,5.585428,5.674181,4.588571,5.184829,5.359352,5.266632,5.406793
A1CF,3.815948,4.142036,4.079639,3.802018,3.970475,4.023918,3.991444,3.832228,3.937628,⋯,4.071126,3.888329,4.035238,3.905470,3.873878,3.941997,4.151041,4.146553,3.663920,3.955250
A2M,7.514131,8.185255,8.142596,8.159722,8.364816,8.133507,7.736251,7.926599,7.856633,⋯,8.756021,8.325534,7.750709,7.957422,7.675870,8.209924,8.338146,8.192327,8.389905,8.364993
A2M-AS1,5.307615,6.067712,6.313713,6.816215,6.534387,6.264997,5.885880,5.029818,6.038523,⋯,6.060199,6.386559,5.476085,6.461962,6.306009,6.154432,7.147076,6.264936,6.224723,4.847976
A2ML1,3.252120,3.498092,3.607268,3.310913,3.254299,3.400641,3.346865,3.238652,3.432481,⋯,3.447296,3.347826,3.241430,3.327122,3.256374,3.429618,3.429215,3.312900,3.188677,3.534706


In [8]:
# access the clinical metadata
clinical <- pData(gse[[1]])

colnames(clinical)
head(clinical)

[1] "title"                          "geo_accession"                 
 [3] "status"                         "submission_date"               
 [5] "last_update_date"               "type"                          
 [7] "channel_count"                  "source_name_ch1"               
 [9] "organism_ch1"                   "characteristics_ch1"           
[11] "characteristics_ch1.1"          "characteristics_ch1.2"         
[13] "characteristics_ch1.3"          "characteristics_ch1.4"         
[15] "characteristics_ch1.5"          "characteristics_ch1.6"         
[17] "characteristics_ch1.7"          "characteristics_ch1.8"         
[19] "characteristics_ch1.9"          "characteristics_ch1.10"        
[21] "characteristics_ch1.11"         "characteristics_ch1.12"        
[23] "characteristics_ch1.13"         "characteristics_ch1.14"        
[25] "characteristics_ch1.15"         "characteristics_ch1.16"        
[27] "characteristics_ch1.17"         "characteristics_ch1.18"        
[29] "treatment_protocol_ch1"         "growth_protocol_ch1"           
[31] "molecule_ch1"                   "extract_protocol_ch1"          
[33] "label_ch1"                      "label_protocol_ch1"            
[35] "taxid_ch1"                      "hyb_protocol"                  
[37] "scan_protocol"                  "description"                   
[39] "data_processing"                "platform_id"                   
[41] "contact_name"                   "contact_email"                 
[43] "contact_laboratory"             "contact_institute"             
[45] "contact_address"                "contact_city"                  
[47] "contact_zip/postal_code"        "contact_country"               
[49] "supplementary_file"             "data_row_count"                
[51] "cellularity:ch1"                "death:ch1"                     
[53] "death_disease_specific:ch1"     "disease status:ch1"            
[55] "drfs_event:ch1"                 "drfs_or_last_contact_years:ch1"
[57] "er:ch1"                         "gender:ch1"                    
[59] "grade:ch1"                      "her2:ch1"                      
[61] "idfs_or_last_contact_years:ch1" "ki67:ch1"                      
[63] "os_or_last_contact_years:ch1"   "pgr:ch1"                       
[65] "positive_nodes:ch1"             "subtype:ch1"                   
[67] "tissue:ch1"                     "tumor type:ch1"

,title,geo_accession,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,characteristics_ch1,⋯,grade:ch1,her2:ch1,idfs_or_last_contact_years:ch1,ki67:ch1,os_or_last_contact_years:ch1,pgr:ch1,positive_nodes:ch1,subtype:ch1,tissue:ch1,tumor type:ch1
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
GSM2345257,PD9888,GSM2345257,Public on Oct 15 2016,Oct 14 2016,Oct 15 2016,RNA,1,"ILC, grade 2",Homo sapiens,disease status: Invasive lobular carcinoma (ILC),⋯,2,Negative,10.9,34,11.8,Positive,8,Trabecular,Breast cancer tumor,ILC
GSM2345258,PD13139,GSM2345258,Public on Oct 15 2016,Oct 14 2016,Oct 15 2016,RNA,1,"ILC, grade 3",Homo sapiens,disease status: Invasive lobular carcinoma (ILC),⋯,3,Negative,9.7,10,10.4,Negative,2,Solid,Breast cancer tumor,ILC
GSM2345259,PD13143,GSM2345259,Public on Oct 15 2016,Oct 14 2016,Oct 15 2016,RNA,1,"ILC, grade 2",Homo sapiens,disease status: Invasive lobular carcinoma (ILC),⋯,2,Negative,11.2,5,12.8,Negative,0,Trabecular,Breast cancer tumor,ILC
GSM2345260,PD13148,GSM2345260,Public on Oct 15 2016,Oct 14 2016,Oct 15 2016,RNA,1,"ILC, grade 2",Homo sapiens,disease status: Invasive lobular carcinoma (ILC),⋯,2,NA,7.3,18,13.7,Positive,0,Classic,Breast cancer tumor,ILC
GSM2345261,PD9867,GSM2345261,Public on Oct 15 2016,Oct 14 2016,Oct 15 2016,RNA,1,"ILC, grade 1",Homo sapiens,disease status: Invasive lobular carcinoma (ILC),⋯,1,Negative,7.9,11,14.8,Positive,2,Classic,Breast cancer tumor,ILC
GSM2345262,PD13155,GSM2345262,Public on Oct 15 2016,Oct 14 2016,Oct 15 2016,RNA,1,"ILC, grade 2",Homo sapiens,disease status: Invasive lobular carcinoma (ILC),⋯,2,Negative,8.6,67.5,8.6,Positive,0,Trabecular,Breast cancer tumor,ILC


In [10]:
t(head(clinical, 1))

,GSM2345257
title,PD9888
geo_accession,GSM2345257
status,Public on Oct 15 2016
submission_date,Oct 14 2016
last_update_date,Oct 15 2016
type,RNA
channel_count,1
source_name_ch1,"ILC, grade 2"
organism_ch1,Homo sapiens
characteristics_ch1,disease status: Invasive lobular carcinoma (ILC)


In [14]:
clinical_filtered <- clinical %>%
  mutate(relapse_free_time = as.integer(as.numeric(`idfs_or_last_contact_years:ch1`) * 365.25)) %>%
  mutate(relapse_free_event = as.integer(`drfs_event:ch1` == "Yes"))

In [ ]:
# Extract relevant columns
clinical_filtered <- clinical_filtered[, c(
  "geo_accession",
  "tumor:ch1",
  "relapse_free_event",
  "relapse_free_time"
)]

# Rename columns
colnames(clinical_filtered) <- c(
  "sample_id",
  "is_tumor",
  "relapse_free_event",
  "relapse_free_time"
)

# encode is_tumor as binary
clinical$is_tumor <- ifelse(clinical$is_tumor == "breast cancer tumor", 1, 0)

cols_to_convert <- c("is_tumor", "relapse_free_event", "relapse_free_time")

In [17]:
head(clinical_filtered)

,sample_id,relapse_free_event,relapse_free_time
,<chr>,<int>,<int>
GSM2345257,GSM2345257,1,3981
GSM2345258,GSM2345258,1,3542
GSM2345259,GSM2345259,0,4090
GSM2345260,GSM2345260,0,2666
GSM2345261,GSM2345261,1,2885
GSM2345262,GSM2345262,0,3141


In [18]:
# save datasets in datasets folder for persistent storage
write.csv(normalized_expr, "../datasets/csv_files/expression_matrix_test_three.csv")
write.csv(clinical_filtered, "../datasets/csv_files/clinical_metadata_test_three.csv")